In [5]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York")

import sys
sys.path.append("../../")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from Query.Base.query_resolution import resolve_query
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue

In [12]:
ts = datetime.date(2026, 3, 23) 
risk = 10_000

mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")
q0 = FixedRateBondQuery(
    cusip="CT5",
    value=FixedRateBondValue.CLEAN_PRICE,
    structure_kwargs={"bpv": risk},
)

req = q0.build_mdp_request(ts)

bond_handle = mdp.get_pricer(dict(req))

q = resolve_query(q0, timestamp=ts, pricer_or_curve=bond_handle)

pkg, rws = q.resolve_package(pricer_or_curve=bond_handle)

vmap = q.build_value_map(
    pricer_or_curve=bond_handle,
    package=pkg,
    risk_weights=rws,
)

print("request:", req)
print("resolved query cusip/label:", [f"{c}: {p._meta_data["cusip"]}, {p._meta_data["label"]}" for c, p in bond_handle.items()])
print("package:", pkg)
print("risk weights:", rws)
print("package notionals:", [f"{p.notional:_.0f}" for p in pkg])

request: {'timestamp': datetime.date(2026, 3, 23), 'cusips': ['CT5']}
resolved query cusip/label: ['CT5: 91282CQD6, T 3 1/2 Feb 31']
package: [FixedRateBondPricableSpec(instrument=<QuantLib.QuantLib.FixedRateBond; proxy of <Swig Object of type 'ext::shared_ptr< FixedRateBond > *' at 0x0000020BB4A90B70> >, cusip='CT5', issue_date=datetime.date(2026, 3, 2), maturity_date=datetime.date(2031, 2, 28), cpn=3.5, notional=22786879.725766618)]
risk weights: [1.0]
package notionals: ['22_786_880']


In [13]:
for v in [
    FixedRateBondValue.YTM,
    FixedRateBondValue.MOD_DURATION,
	FixedRateBondValue.CARRY_BPS_RUNNING,
	FixedRateBondValue.ROLL_BPS_RUNNING,
]:
    print(v.name, vmap.apply(value=v))

YTM 3.956
MOD_DURATION 4.469619087952642
CARRY_BPS_RUNNING -0.180369352206975
ROLL_BPS_RUNNING 1.5624874147267764
